In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.adk.agents import Agent, LoopAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [3]:
#Importing Memory Components
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory, preload_memory
from google.genai import types

print("✅ ADK memory components imported successfully.")

✅ ADK memory components imported successfully.


In [4]:
async def run_session(
    runner_instance: Runner, user_queries: list[str] | str, session_id: str = "default"
):
    """Helper function to run queries in a session and display responses."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    # Convert single query to list
    if isinstance(user_queries, str):
        user_queries = [user_queries]

 # Process each query
    for query in user_queries:
        print(f"\nUser > {query}")
        query_content = types.Content(role="user", parts=[types.Part(text=query)])
        # Stream agent response
        async for event in runner_instance.run_async(
            user_id=USER_ID, session_id=session.id, new_message=query_content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model: > {text}")


print("✅ Functions to run each session and process a query is defined.")

✅ Functions to run each session and process a query is defined.


In [5]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [6]:
#Defining an agent to identify the genre from user prompt

genre_identifier_agent = LlmAgent(
    name="GenreIdentifier",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
You are a blog genre classifier. 
Input: User's blog topic or description.
Output: Identify the most appropriate genre from:
Health, News, BibleApp, Fashion, Business, Tech
Respond only with the genre as a single word.
""" ,
    output_key= "genre"
)
print("✅ Genre Identifier Agent created.")


✅ Genre Identifier Agent created.


In [7]:
#Initial writer agent generates draft blog based on requirements obtained from Memory retrieval agent and using google search as tool.

initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),

    tools=[google_search],

    instruction="""
You create the first blog draft.

Inputs:
- topic (from user)
- genre (from genre_identifier_agent)

JSON Input: {genre}

""",

    output_key="current_version"
)

print("✅ InitialWriterAgent created.")


✅ InitialWriterAgent created.


In [8]:
# Critic Agent evaluates the blog based on genre-specific characteristics.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
You are a constructive blog critic. Review the blog provided below.

The input is a JSON with:
- "genre": the blog genre
- "draft": the full blog text

JSON Input: {current_version}

Evaluate the blog based on the following genre-specific characteristics:

- Health: Word count 800–1500, 8–15 paragraphs, semi-formal tone, headings/subheadings, bullet points, lists, call-to-action, charts/images.
- News: Word count 400–1000, 5–10 paragraphs, formal tone, headline → lead → body → quotes → inverted pyramid, bullet points for key facts.
- BibleApp/Religious: Word count 500–1200, 6–12 paragraphs, informal/devotional tone, headings, scripture citations, bullet points, quotes, reflective conclusion.
- Fashion: Word count 600–1200, 6–12 paragraphs, informal/trendy tone, headings/subheadings, bullet points, images, call-to-action.
- Business: Word count 800–2000, 8–18 paragraphs, formal/semi-formal tone, headings/subheadings, problem→analysis→solution structure, charts/graphs, bullet points, case studies, conclusion.
- Tech: Word count 800–2000, 8–20 paragraphs, semi-formal/formal tone, headings/subheadings, features/comparison, pros/cons, bullet points/lists, code snippets/diagrams, conclusion.
Instructions for review:
- Check readability, factual accuracy, structure, word count, paragraph count, tonality, and format as per the genre.
- If the blog meets these above given set of criteria and is grammatically and factually correct, respond with the exact phrase: "APPROVED".
- Otherwise, provide 2–3 specific actionable suggestions for improvement.Give the feedback in  precise bullet points.
""",
    output_key="critique",
)

print("✅ critic_agent created.")


✅ critic_agent created.


In [9]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the blog is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


In [10]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a blog refiner. You have a blog draft and critique.
    
    Blog Draft: {current_version}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.
    - Your response should only be the final output blog""",
    output_key="current_version",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")


✅ refiner_agent created.


In [11]:
from pydantic import BaseModel, Field
from typing import Literal

# Structured output schema so the judge's scores are consistently parseable
class EvaluationResult(BaseModel):
    factuality_score: int = Field(description="Score 1-5 for factual accuracy and absence of hallucinated claims")
    factuality_reasoning: str = Field(description="Brief justification for the factuality score")

    instruction_adherence_score: int = Field(description="Score 1-5 for how well the blog follows the original topic/instructions/requirements")
    instruction_adherence_reasoning: str = Field(description="Brief justification for the instruction adherence score")

    style_format_score: int = Field(description="Score 1-5 for tone, readability, structure, and formatting quality")
    style_format_reasoning: str = Field(description="Brief justification for the style/format score")

    overall_verdict: Literal["PASS", "FAIL"] = Field(description="PASS if all scores >= 3, otherwise FAIL")
    summary: str = Field(description="1-2 sentence overall assessment")


# This agent evaluates the final blog output against fixed criteria and produces a structured verdict.
judge_agent = Agent(
    name="JudgeAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are an impartial LLM judge evaluating a generated blog post.

    Original Topic / Instructions: user input
    Final Blog Draft: {current_version}

    Evaluate the blog strictly on the following three criteria, each scored 1-5 (5 = excellent, 1 = very poor):

    1. FACTUALITY
       - Are claims accurate and verifiable, or at least plausible and non-misleading?
       - Flag any hallucinated facts, statistics, names, or events not grounded in reality or provided context.

    2. INSTRUCTION ADHERENCE
       - Does the blog fully address the original topic/instructions?
       - Does it respect any constraints (length, audience, tone, structure) implied by the request?

    3. STYLE AND FORMAT
       - Is the writing clear, engaging, and well-organized (headings, flow, paragraphing)?
       - Is the tone appropriate for a blog post (not robotic, not overly verbose)?

    Be strict and objective. Do not inflate scores. Justify each score in 1-2 sentences.
    Set overall_verdict to PASS only if ALL THREE scores are 3 or higher; otherwise FAIL.

    Return ONLY the structured evaluation output, nothing else.""",
    output_key="evaluation_result",
    output_schema=EvaluationResult,
)
print("✅ judge_agent created.")

✅ judge_agent created.


In [12]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
blog_refinement_loop = LoopAgent(
    name="BlogRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[genre_identifier_agent,initial_writer_agent, blog_refinement_loop,judge_agent],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


/tmp/ipykernel_49/1933743406.py:2: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  blog_refinement_loop = LoopAgent(
/tmp/ipykernel_49/1933743406.py:9: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [14]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "AI Engineers are the new Software Engineers"
)


GenreIdentifier > Tech
InitialWriterAgent > The landscape of software engineering is rapidly evolving, with Artificial Intelligence (AI) Engineers emerging as a critical and highly sought-after specialization. While traditional software engineers focus on building deterministic systems with explicit logic, AI engineers develop probabilistic learning systems that improve with new data. This distinction is leading to a surge in demand for AI engineers, with some reports even ranking "Artificial Intelligence Engineer" as the number one fastest-growing job.

AI engineers are responsible for tasks such as model training, algorithm development, and continuous model retraining. They leverage machine learning (ML) algorithms and deep neural networks (DNN) to extract business insights and build intelligent systems. Key responsibilities include model development, validation, testing, and deploying these models into production. Furthermore, AI engineers often build the infrastructure for data ing